# Deep EDA and Classification: Framing the ML Problem
In this module, we will explore the Volve Production dataset in depth. 
Instead of predicting an exact continuous number (Regression), we will reframe our business 
logic to predict categories (Classification). 

We will tackle:
1. **Extensive Exploratory Data Analysis (EDA)**
2. **Binary Classification:** Is the well highly productive today? (Yes/No)
3. **Multi-class Classification:** What tier of production are we in? (Low/Medium/High)

In [ ]:
# Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import Scikit-Learn modules
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Import Classification Models & Metrics
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Set plotting style
sns.set_theme(style="whitegrid", context="notebook")

## 1. Data Loading & Feature Engineering
We will load the data and immediately create our categorical targets.

In [ ]:
file_path = "Volve_Production_Data.csv" 
df = pd.read_csv(file_path)

# Keep only active production days
df = df.dropna(subset=['BORE_OIL_VOL'])
df = df[df['BORE_OIL_VOL'] > 0].copy()

# Feature Engineering: Creating Categorical Targets
median_vol = df['BORE_OIL_VOL'].median()
quantiles = df['BORE_OIL_VOL'].quantile([0.33, 0.66]).to_dict()

# Target 1: Binary (1 = High Producer, 0 = Low Producer)
df['Target_Binary'] = (df['BORE_OIL_VOL'] > median_vol).astype(int)

# Target 2: Multi-class (0 = Low, 1 = Medium, 2 = High)
def categorize_production(vol):
    if vol <= quantiles[0.33]: return 'Low'
    elif vol <= quantiles[0.66]: return 'Medium'
    else: return 'High'

df['Target_Multi'] = df['BORE_OIL_VOL'].apply(categorize_production)

# Define our feature space
features = [
    'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 
    'AVG_ANNULUS_PRESS', 'AVG_CHOKE_SIZE_P', 'AVG_WHP_P', 'AVG_WHT_P'
]

## 2. Exploratory Data Analysis (EDA)
Before feeding data to a model, we must understand its distribution, outliers, and relationships.

In [ ]:
# EDA 1: Feature Distributions
# Are our sensors recording normally distributed data, or is it skewed?
plt.figure(figsize=(15, 10))
for i, feature in enumerate(features, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[feature], bins=30, kde=True, color='teal')
    plt.title(f'Distribution: {feature}', fontsize=10)
    plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# EDA 2: Target Class Balance
# Imbalanced classes can destroy a model. Let's check our multi-class target.
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Target_Multi', order=['Low', 'Medium', 'High'], palette='viridis')
plt.title('Distribution of Production Tiers (Multi-class)')
plt.ylabel('Number of Days')
plt.show()

In [ ]:
# EDA 3: Feature vs. Target Relationships (Boxplots)
# How do specific features behave across our production tiers?
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='Target_Multi', y='AVG_CHOKE_SIZE_P', order=['Low', 'Medium', 'High'])
plt.title('Choke Size vs Production Tier')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Target_Multi', y='AVG_DOWNHOLE_PRESSURE', order=['Low', 'Medium', 'High'])
plt.title('Downhole Pressure vs Production Tier')

plt.tight_layout()
plt.show()

## 3. Pipeline A: Binary Classification
**Goal:** Predict if the well is producing above the median volume today (`Target_Binary`).
We will use Logistic Regression, which is highly suited for binary outputs (0 or 1).

In [ ]:
X = df[features]
y_bin = df['Target_Binary']

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(X, y_bin, test_size=0.25, random_state=42)

# Build and train the Binary Pipeline
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

binary_pipeline.fit(X_train_bin, y_train_bin)
bin_preds = binary_pipeline.predict(X_test_bin)

# Evaluate
print("--- Binary Classification Report (Logistic Regression) ---")
print(classification_report(y_test_bin, bin_preds, target_names=['Low Producer', 'High Producer']))

# Confusion Matrix Visual
cm_bin = confusion_matrix(y_test_bin, bin_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_bin, display_labels=['Low', 'High'])
disp.plot(cmap='Blues')
plt.title("Binary Confusion Matrix")
plt.show()

## 4. Pipeline B: Multi-class Classification
**Goal:** Predict the specific tier of production: Low, Medium, or High (`Target_Multi`).
We will use a Decision Tree, which natively handles multi-class data by branching.

In [ ]:
y_multi = df['Target_Multi']

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(X, y_multi, test_size=0.25, random_state=42)

# Build and train the Multi-class Pipeline
# Note: Decision Trees don't strictly require scaling, but imputation is still necessary.
multi_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42)) # Limit depth to avoid overfitting
])

multi_pipeline.fit(X_train_multi, y_train_multi)
multi_preds = multi_pipeline.predict(X_test_multi)

# Evaluate
print("--- Multi-class Classification Report (Decision Tree) ---")
print(classification_report(y_test_multi, multi_preds, labels=['Low', 'Medium', 'High']))

# Confusion Matrix Visual
cm_multi = confusion_matrix(y_test_multi, multi_preds, labels=['Low', 'Medium', 'High'])
disp_multi = ConfusionMatrixDisplay(confusion_matrix=cm_multi, display_labels=['Low', 'Medium', 'High'])
disp_multi.plot(cmap='Greens')
plt.title("Multi-class Confusion Matrix")
plt.show()

## 5. Decision Tree Logic Map
One of the biggest advantages of a Decision Tree for multi-class problems is interpretability.
We can visualize exactly how it separates the Low, Medium, and High tiers.

In [ ]:
tree_model = multi_pipeline.named_steps['model']

plt.figure(figsize=(20, 10))
plot_tree(
    tree_model, 
    feature_names=features, 
    class_names=['High', 'Low', 'Medium'], # Align with model's internal class ordering
    filled=True, 
    rounded=True, 
    fontsize=9
)
plt.title("Multi-class Decision Tree Logic")
plt.show()